# AutoSchemaKG — HotpotQA English 100-question pilot

Seeded 100-question `distractor/validation` run with durable Drive checkpoints. Change `RUN_PHASE`, then run all cells. Recommended order: `prepare`, repeat `extract`, `build`, `package`, `benchmark`, `report`. This is a pilot, not the paper's 1,000-question reproduction.


In [ ]:
#@title 1. Configuration
RUN_PHASE = 'prepare' #@param ['prepare', 'extract', 'build', 'package', 'benchmark', 'report']
MAX_EXTRACTION_CHUNKS = 100 #@param {type:'integer'}
FILTER_FAILURE_POLICY = 'error' #@param ['error', 'dense']

import os, sys, json, shutil, subprocess, time
from pathlib import Path
import requests
from google.colab import drive

drive.mount('/content/drive')
RUN_ROOT = Path('/content/drive/MyDrive/AutoSchemaKG/hotpotqa_en_100q_seed42')
RUN_ROOT.mkdir(parents=True, exist_ok=True)
WORK_DIR = RUN_ROOT / 'experiment'
CODE_REF = 'codex/research-concept-retrieval'
MODEL_ID = 'Qwen/Qwen3.5-2B'
EMBEDDING_MODEL = 'sentence-transformers/multi-qa-MiniLM-L6-cos-v1'
PORT = 8000
CONTEXT_LENGTH = 4096
NEEDS_LLM = RUN_PHASE in {'extract', 'build', 'benchmark'}
if NEEDS_LLM:
    gpu = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    print(gpu.stdout)
    if gpu.returncode != 0:
        raise RuntimeError('Select Runtime > Change runtime type > GPU')
print('Phase:', RUN_PHASE, 'Persistent root:', RUN_ROOT, 'Needs LLM:', NEEDS_LLM)


## 2. Clone and pin code/model revisions
The first run pins the Git commit and Hugging Face revisions in Drive. Later sessions restore exactly those revisions.


In [ ]:
REPO_DIR = Path('/content/SmallScaledAutoSchemaKG_100q')
# Split the scheme so notebook/link renderers cannot rewrite this into [URL](URL).
REPO_URL = 'https://' + 'github.com/phuongth05/SmallScaledAutoSchemaKG.git'
if REPO_URL.startswith('[') or '](' in REPO_URL:
    raise ValueError(f'REPO_URL must be a plain URL, got: {REPO_URL!r}')
# GitHub API/smart-HTTP diagnostics showed that only the inherited runtime
# HOME/environment fails; an isolated HOME succeeds. Keep this scoped to Git.
GIT_CLEAN_HOME = Path('/tmp/autoschema_git_home')
GIT_CLEAN_HOME.mkdir(parents=True, exist_ok=True)
# Use the exact minimal environment proven by the diagnostic probe.
# Do not start from os.environ.copy(): that retained the runtime contaminant.
GIT_ENV = {
    'PATH': os.environ['PATH'],
    'HOME': str(GIT_CLEAN_HOME),
    'LANG': 'C.UTF-8',
    'LC_ALL': 'C.UTF-8',
    'GIT_TERMINAL_PROMPT': '0',
    'GIT_CONFIG_NOSYSTEM': '1',
    'GIT_CONFIG_GLOBAL': '/dev/null',
}
for name in ('SSL_CERT_FILE', 'SSL_CERT_DIR'):
    if os.environ.get(name):
        GIT_ENV[name] = os.environ[name]
if REPO_DIR.exists() and not (REPO_DIR / '.git').exists():
    backup = REPO_DIR.with_name(REPO_DIR.name + f'_failed_clone_{int(time.time())}')
    shutil.move(str(REPO_DIR), str(backup))
    print('Preserved incomplete previous clone at:', backup)
if not (REPO_DIR / '.git').exists():
    probe = subprocess.run(['git', 'ls-remote', '--exit-code', '--heads', REPO_URL, CODE_REF], capture_output=True, text=True, env=GIT_ENV)
    if probe.returncode != 0:
        raise RuntimeError(f'GitHub probe failed ({probe.returncode}). URL={REPO_URL!r}\n{probe.stdout}\n{probe.stderr}')
    clone = subprocess.run(['git', 'clone', '--single-branch', '--branch', CODE_REF, REPO_URL, str(REPO_DIR)], capture_output=True, text=True, env=GIT_ENV)
    print(clone.stdout, clone.stderr, sep='\n')
    if clone.returncode != 0:
        raise RuntimeError(f'git clone failed ({clone.returncode}); see output above')
revision_file = RUN_ROOT / 'revisions.json'
if revision_file.exists():
    revisions = json.loads(revision_file.read_text())
    subprocess.run(['git', 'checkout', '--detach', revisions['git_commit']], cwd=REPO_DIR, check=True, env=GIT_ENV)
    build_script = REPO_DIR / 'scripts/run_colab_v1.py'
    if 'load_build_progress' not in build_script.read_text(encoding='utf-8'):
        old_commit = revisions['git_commit']
        subprocess.run(['git', 'fetch', 'origin', CODE_REF], cwd=REPO_DIR, check=True, env=GIT_ENV)
        subprocess.run(['git', 'checkout', '--detach', 'FETCH_HEAD'], cwd=REPO_DIR, check=True, env=GIT_ENV)
        if 'load_build_progress' not in build_script.read_text(encoding='utf-8'):
            raise RuntimeError('The research branch does not contain resumable build yet')
        revisions.setdefault('code_upgrade_history', []).append(old_commit)
        revisions['git_commit'] = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
        revision_file.write_text(json.dumps(revisions, indent=2))
        print('Upgraded code for resumable build:', old_commit, '->', revisions['git_commit'])
else:
    subprocess.run(['git', 'fetch', 'origin', CODE_REF], cwd=REPO_DIR, check=True, env=GIT_ENV)
    subprocess.run(['git', 'checkout', '--detach', 'FETCH_HEAD'], cwd=REPO_DIR, check=True, env=GIT_ENV)
    if not (REPO_DIR / 'scripts/run_hotpotqa_en.py').is_file():
        raise RuntimeError('The selected revision does not contain the 100-question pipeline')
    def model_revision(model):
        response = requests.get(f'https://huggingface.co/api/models/{model}', timeout=30)
        response.raise_for_status()
        return response.json()['sha']
    revisions = {
        'git_commit': subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip(),
        'model': MODEL_ID, 'model_revision': model_revision(MODEL_ID),
        'embedding': EMBEDDING_MODEL, 'embedding_revision': model_revision(EMBEDDING_MODEL),
    }
    revision_file.write_text(json.dumps(revisions, indent=2))
if revisions['model'] != MODEL_ID or revisions['embedding'] != EMBEDDING_MODEL:
    raise ValueError('Saved models differ; use a new RUN_ROOT')
os.chdir(REPO_DIR)
print(json.dumps(revisions, indent=2))


## 3. Isolated environments
QA/construction uses CPU Torch; vLLM has a separate GPU environment to avoid Torch/TorchAudio CUDA conflicts. Installation logs and locks persist on Drive.


In [ ]:
from collections import deque
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'uv'], check=True)
QA_ENV = Path('/content/autoschema_100q_qa_env')
VLLM_ENV = Path('/content/autoschema_100q_vllm_env')
QA_PY = str(QA_ENV / 'bin/python')
VLLM_PY = str(VLLM_ENV / 'bin/python')

def run_logged(label, command):
    log_path = RUN_ROOT / f'{label}.log'
    recent = deque(maxlen=80)
    print('\n===', label, '===', flush=True)
    with log_path.open('a', encoding='utf-8') as log_file:
        process = subprocess.Popen(list(map(str, command)), cwd=str(REPO_DIR), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace', bufsize=1)
        for line in process.stdout:
            print(line, end='', flush=True); log_file.write(line); log_file.flush(); recent.append(line.rstrip())
        code = process.wait()
    if code:
        raise RuntimeError(f'{label} failed ({code}). Log: {log_path}\n' + '\n'.join(recent))

if not Path(QA_PY).exists():
    subprocess.run(['uv', 'venv', '--python', '3.12', str(QA_ENV)], check=True)
qa_lock = RUN_ROOT / 'qa_requirements.lock.txt'
if qa_lock.exists() and qa_lock.stat().st_size:
    run_logged('qa_lock_install', ['uv', 'pip', 'install', '--python', QA_PY, '--torch-backend=cpu', '-r', qa_lock])
else:
    run_logged('qa_base_install', ['uv', 'pip', 'install', '--python', QA_PY, '--torch-backend=cpu', '-r', REPO_DIR / 'requirements-hotpotqa-v2.txt'])
    run_logged('qa_repo_install', ['uv', 'pip', 'install', '--python', QA_PY, '--torch-backend=cpu', '--constraint', REPO_DIR / 'requirements-hotpotqa-v2.txt', '-r', REPO_DIR / 'requirements-colab.txt'])
    with qa_lock.open('w', encoding='utf-8') as stream:
        subprocess.run(['uv', 'pip', 'freeze', '--python', QA_PY], cwd=REPO_DIR, stdout=stream, text=True, check=True)
if NEEDS_LLM:
    if not Path(VLLM_PY).exists():
        subprocess.run(['uv', 'venv', '--python', '3.12', str(VLLM_ENV)], check=True)
    vllm_lock = RUN_ROOT / 'vllm_requirements.lock.txt'
    vllm_args = ['-r', vllm_lock] if vllm_lock.exists() and vllm_lock.stat().st_size else ['--pre', 'vllm']
    run_logged('vllm_install', ['uv', 'pip', 'install', '--python', VLLM_PY, '--torch-backend=auto', *vllm_args])
    if not vllm_lock.exists() or not vllm_lock.stat().st_size:
        with vllm_lock.open('w', encoding='utf-8') as stream:
            subprocess.run(['uv', 'pip', 'freeze', '--python', VLLM_PY], cwd=REPO_DIR, stdout=stream, text=True, check=True)
subprocess.run([QA_PY, '-c', "import torch; print('QA torch:', torch.__version__)"], check=True)


## 4. Start local Qwen server when required
Skip automatically for prepare/package/report. Build includes LLM concept induction, so extract, build and benchmark require the server. After a runtime reset, start it again.


In [ ]:
def server_ready():
    try:
        response = requests.get(f'http://127.0.0.1:{PORT}/v1/models', timeout=5)
        return response.ok and any(item['id'] == MODEL_ID for item in response.json().get('data', []))
    except requests.RequestException:
        return False

if NEEDS_LLM and not server_ready():
    VLLM_BIN = VLLM_ENV / 'bin/vllm'
    if not VLLM_BIN.is_file():
        raise FileNotFoundError(f'vLLM executable missing: {VLLM_BIN}')
    LOG_PATH = RUN_ROOT / 'qwen_vllm.log'
    server_log = LOG_PATH.open('a', encoding='utf-8')
    server = subprocess.Popen([str(VLLM_BIN), 'serve', MODEL_ID, '--revision', revisions['model_revision'], '--host', '127.0.0.1', '--port', str(PORT), '--dtype', 'half', '--max-model-len', str(CONTEXT_LENGTH), '--max-num-seqs', '1'], stdout=server_log, stderr=subprocess.STDOUT)
    deadline = time.time() + 900
    while time.time() < deadline and not server_ready():
        if server.poll() is not None:
            server_log.flush()
            tail = LOG_PATH.read_text(encoding='utf-8', errors='replace')[-8000:]
            raise RuntimeError('vLLM stopped early:\n' + tail)
        time.sleep(5)
    if not server_ready():
        raise TimeoutError(f'vLLM not ready; inspect {LOG_PATH}')
print('Server required:', NEEDS_LLM, 'Ready:', server_ready() if NEEDS_LLM else 'skipped')


## 5. Run selected phase
For extraction, rerun this cell until `Extraction complete: True`. Benchmark also resumes completed `(method, question)` checkpoints. Use `FILTER_FAILURE_POLICY='error'` for reportable results.


In [ ]:
command = [QA_PY, '-X', 'utf8', '-u', 'scripts/run_hotpotqa_en.py', '--phase', RUN_PHASE, '--work-dir', str(WORK_DIR), '--max-questions', '100', '--sampling', 'random', '--seed', '42', '--model', MODEL_ID, '--model-revision', revisions['model_revision'], '--embedding-model', EMBEDDING_MODEL, '--embedding-revision', revisions['embedding_revision'], '--base-url', f'http://127.0.0.1:{PORT}/v1', '--context-length', str(CONTEXT_LENGTH), '--filter-failure-policy', FILTER_FAILURE_POLICY]
if RUN_PHASE == 'extract':
    command += ['--max-extraction-chunks', str(MAX_EXTRACTION_CHUNKS)]
print('Running:', ' '.join(command), flush=True)
run_logged('phase_' + RUN_PHASE, command)
print('Extraction complete:', (WORK_DIR / 'graph/extraction_complete.json').exists())
print('Build complete:', (WORK_DIR / 'graph/build_complete.json').exists())
summary = WORK_DIR / 'benchmark/summary.json'
if summary.exists():
    print(summary.read_text(encoding='utf-8'))


## 6. Inspect saved report/artifacts


In [ ]:
for relative in ['construction.zip', 'benchmark/summary.json', 'benchmark/evaluation_100q.md', 'benchmark/evaluation_100q.json']:
    path = WORK_DIR / relative
    print(relative, 'exists:', path.exists(), 'bytes:', path.stat().st_size if path.exists() else None)
report = WORK_DIR / 'benchmark/evaluation_100q.md'
if report.exists():
    from IPython.display import Markdown, display
    display(Markdown(report.read_text(encoding='utf-8')))
